# Why do we need a separate validation set for hyperparameter tuning? Write your answer in text.

Validation set is a piece of the data that is used before testing the model, and its purpose is model selection, hypermarameter tuning, it is done by first raining the model on the training dataset which gets the values for the parameters (coofficients), and then the validation set is used on the same trained model with different hyperparameters and different values for them to decide which would result with the best result. Then after both training and validation (parameter and hyperparameter) are done then the test set is used to test the model on entirely unseen and unused data. We cannot use the test set for this portion because it would result in what is called data leakage meaning the optimised model we chose fits the test data in the terms of the best possible hyperparameters for the test data meaning the testing data is not data that is unseen before to the model.

# List 2 hyperparameter tuning methods. Write your answer in text.

Hyperparameter tuning is made up of two phases first is the search method of hyperparameter values and the second phase is evaluation of the resulting values of the hyperparameters.

Two methods of searching for hyperparameter values are:

1. Grid Search which works by you giving a set of values you want to test for each hyperparameter and the grid search tries all different combinations of those values, so you can see every possible output for those values that you gave.

2. Random Search which works by you giving a range for a hyperparameter to be in and it samples values from that range for each, it works by you giving a niter variable which means how many iterations you want to be done.

The Random search seems bad but actually if you gave a grid search two sets of 5 values for two hyper parameters even though there is 25 possible values it still only tried 5 values for each, while random search potentially could try 25 different values for each, meaning it can try more possiblities with in the same time.

Evaluation part can be done by any metric but most prominantly is the F1 score on the validation set to judge which values are best for the hyperparamters.

## Hyperparameter Search with Train, Validation, and Test Sets

F1 score is the model-selection metric because it balances positive-class precision and recall. The validation set is used to choose hyperparameters, while the test set remains untouched until the final evaluation.


In [2]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split

# Reuse the same custom preprocessing, tokenization, and TF-IDF vectorization.
current_path = Path.cwd().resolve()
repository_root = next(
    (path for path in (current_path, *current_path.parents)
     if (path / "phase-1-machine-learning-nlp").is_dir()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("Could not locate the repository root.")

phase_1 = repository_root / "phase-1-machine-learning-nlp"
part_02 = phase_1 / "02-preprocessing-tokenization"
part_06 = phase_1 / "06-vectorization"
sys.path.insert(0, str(part_02 / "src"))
sys.path.insert(0, str(part_06 / "src"))

from preprocessor import preprocess_text
from tokenizer import regex_tokenize
from vectorizer import CustomTfidfVectorizer

# Prepare the IMDB data exactly as before.
imdb = pd.read_csv(part_02 / "data" / "IMDB Dataset.csv")
imdb = imdb.dropna(subset=["review", "sentiment"])
imdb = imdb.drop_duplicates(subset="review").reset_index(drop=True)
imdb["label"] = imdb["sentiment"].map({"negative": 0, "positive": 1})

# First reserve 20% for testing, then split the remaining 80% into 60%/20%.
X_development, X_test, y_development, y_test = train_test_split(
    imdb["review"], imdb["label"], test_size=0.20,
    random_state=42, stratify=imdb["label"],
)
X_train, X_validation, y_train, y_validation = train_test_split(
    X_development, y_development, test_size=0.25,
    random_state=42, stratify=y_development,
)

# Fit the vectorizer on training text only to avoid validation/test leakage.
text_vectorizer = CustomTfidfVectorizer(
    preprocessor=preprocess_text, tokenizer=regex_tokenize,
    min_df=5, max_df=0.95, max_features=30_000,
)
X_train_tfidf = text_vectorizer.fit_transform(X_train)
X_validation_tfidf = text_vectorizer.transform(X_validation)
X_test_tfidf = text_vectorizer.transform(X_test)

print(f"Training reviews: {len(X_train):,}")
print(f"Validation reviews: {len(X_validation):,}")
print(f"Test reviews: {len(X_test):,}")
print(f"TF-IDF features: {X_train_tfidf.shape[1]:,}")


Training reviews: 29,748
Validation reviews: 9,917
Test reviews: 9,917
TF-IDF features: 30,000


In [3]:
# Build a small manual search space for the Topic 9 regularizers.
search_configurations = []
for C in [0.1, 1.0, 4.0]:
    search_configurations.append({
        "regularization": "L2", "C": C,
        "l1_ratio": 0.0, "solver": "lbfgs",
    })
    search_configurations.append({
        "regularization": "L1", "C": C,
        "l1_ratio": 1.0, "solver": "liblinear",
    })

for C in [0.1, 1.0, 4.0]:
    for l1_ratio in [0.25, 0.50, 0.75]:
        search_configurations.append({
            "regularization": "Elastic Net", "C": C,
            "l1_ratio": l1_ratio, "solver": "saga",
        })

search_rows = []
trained_models = []
for configuration in search_configurations:
    candidate_model = LogisticRegression(
        C=configuration["C"],
        l1_ratio=configuration["l1_ratio"],
        solver=configuration["solver"],
        max_iter=2_000,
        random_state=42,
    )
    candidate_model.fit(X_train_tfidf, y_train)
    validation_predictions = candidate_model.predict(X_validation_tfidf)

    search_rows.append({
        **configuration,
        "validation accuracy": accuracy_score(
            y_validation, validation_predictions
        ),
        "validation precision": precision_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "validation recall": recall_score(
            y_validation, validation_predictions, zero_division=0
        ),
        "validation F1": f1_score(
            y_validation, validation_predictions, zero_division=0
        ),
    })
    trained_models.append(candidate_model)

search_results = pd.DataFrame(search_rows)
best_result_index = search_results["validation F1"].idxmax()
best_configuration = search_results.loc[best_result_index]
best_model = trained_models[best_result_index]

display(
    search_results.sort_values("validation F1", ascending=False)
    .reset_index(drop=True)
    .style.format({
        "C": "{:.2f}", "l1_ratio": "{:.2f}",
        "validation accuracy": "{:.4f}",
        "validation precision": "{:.4f}",
        "validation recall": "{:.4f}",
        "validation F1": "{:.4f}",
    })
)


,regularization,C,l1_ratio,solver,validation accuracy,validation precision,validation recall,validation F1
0,Elastic Net,4.00,0.75,saga,0.8843,0.8727,0.9009,0.8866
1,Elastic Net,4.00,0.25,saga,0.8837,0.8733,0.8987,0.8858
2,Elastic Net,4.00,0.50,saga,0.8834,0.8719,0.8999,0.8857
3,L2,4.00,0.00,lbfgs,0.8832,0.8739,0.8967,0.8852
4,L1,4.00,1.00,liblinear,0.8830,0.8725,0.8981,0.8851
5,L2,1.00,0.00,lbfgs,0.8671,0.8546,0.8859,0.8700
6,Elastic Net,1.00,0.25,saga,0.8626,0.8466,0.8869,0.8663
7,Elastic Net,1.00,0.50,saga,0.8602,0.8435,0.8859,0.8642
8,Elastic Net,1.00,0.75,saga,0.8574,0.8384,0.8869,0.8619
9,L1,1.00,1.00,liblinear,0.8544,0.8377,0.8805,0.8585


In [4]:
# Summarize the validation-F1 winner without consulting the test labels.
best_hyperparameters = pd.DataFrame([{
    "best regularization": best_configuration["regularization"],
    "best C": best_configuration["C"],
    "best l1_ratio": best_configuration["l1_ratio"],
    "solver": best_configuration["solver"],
    "validation F1": best_configuration["validation F1"],
}])
print("Best hyperparameters selected using validation F1:")
display(best_hyperparameters.style.format({
    "best C": "{:.2f}", "best l1_ratio": "{:.2f}",
    "validation F1": "{:.4f}",
}))

# Build the matching Topic 9 default as the before-tuning comparison.
untuned_regularization = best_configuration["regularization"]
untuned_l1_ratio = (
    0.0 if untuned_regularization == "L2"
    else 1.0 if untuned_regularization == "L1"
    else 0.5
)
untuned_solver = (
    "lbfgs" if untuned_regularization == "L2"
    else "liblinear" if untuned_regularization == "L1"
    else "saga"
)
untuned_model = LogisticRegression(
    C=1.0, l1_ratio=untuned_l1_ratio, solver=untuned_solver,
    max_iter=2_000, random_state=42,
)
untuned_model.fit(X_train_tfidf, y_train)

def classification_metrics(model, features, labels):
    predictions = model.predict(features)
    return pd.Series({
        "accuracy": accuracy_score(labels, predictions),
        "positive precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "positive recall": recall_score(
            labels, predictions, zero_division=0
        ),
        "positive F1": f1_score(labels, predictions, zero_division=0),
    })

before_tuning = classification_metrics(untuned_model, X_test_tfidf, y_test)
after_tuning = classification_metrics(best_model, X_test_tfidf, y_test)
test_metrics_comparison = pd.DataFrame({
    "before tuning": before_tuning,
    "after tuning": after_tuning,
})
test_metrics_comparison["change"] = after_tuning - before_tuning

print("Test metrics before and after tuning:")
display(test_metrics_comparison.style.format("{:.4f}"))


Best hyperparameters selected using validation F1:


,best regularization,best C,best l1_ratio,solver,validation F1
0,Elastic Net,4.00,0.75,saga,0.8866


Test metrics before and after tuning:


,before tuning,after tuning,change
accuracy,0.8605,0.8853,0.0248
positive precision,0.8412,0.8751,0.0339
positive recall,0.8901,0.8999,0.0098
positive F1,0.8650,0.8874,0.0224


## Stratified K-Fold Cross-Validation of the Selected Model

The following experiment keeps the hyperparameters selected above and changes only the number of cross-validation folds. F1 remains the evaluation metric, and the reserved test set is not used during cross-validation.


In [ ]:
import numpy as np
from scipy.sparse import vstack
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Reuse the TF-IDF matrices prepared in the previous section.
X_cv = vstack([X_train_tfidf, X_validation_tfidf])
y_cv = pd.concat([y_train, y_validation])

# Reuse the validation-F1 winner; no additional hyperparameter search occurs.
selected_cv_model = LogisticRegression(
    C=float(best_configuration["C"]),
    l1_ratio=float(best_configuration["l1_ratio"]),
    solver=best_configuration["solver"],
    max_iter=2_000,
    random_state=42,
)

cv_rows = []
for k in [3, 5, 10]:
    folds = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    fold_f1_scores = cross_val_score(
        selected_cv_model,
        X_cv,
        y_cv,
        cv=folds,
        scoring="f1",
        n_jobs=1,
        error_score="raise",
    )
    cv_rows.append({
        "k": k,
        "fold F1 scores": np.round(fold_f1_scores, 4).tolist(),
        "mean F1": fold_f1_scores.mean(),
        "standard deviation": fold_f1_scores.std(),
    })




cross_validation_results = pd.DataFrame(cv_rows).set_index("k")
display(cross_validation_results.style.format({
    "mean F1": "{:.4f}",
    "standard deviation": "{:.4f}",
}))


,fold F1 scores,mean F1,standard deviation
k,,,
3,"[0.884, 0.8814, 0.8905]",0.8853,0.0038
5,"[0.888, 0.8796, 0.8866, 0.8871, 0.8921]",0.8867,0.0041
10,"[0.8808, 0.8968, 0.8869, 0.8747, 0.8829, 0.8888, 0.8909, 0.8911, 0.8884, 0.8984]",0.8880,0.0068


### Cross-Validation Observations

- The mean F1 scores were 0.8834 for `k=3`, 0.8867 for `k=5`, and 0.8883 for `k=10`. The total difference between the lowest and highest mean was approximately 0.0049, so the average F1 changed only slightly as `k` increased.
- `k=3` had the lowest standard deviation at 0.0011 and therefore produced the most stable fold scores in this experiment. `k=5` was almost equally stable, also with a standard deviation of approximately 0.0011.
- The standard deviation increased to 0.0043 for `k=10`, even though its mean F1 was the highest. Its individual fold scores varied more than those for `k=3` and `k=5`.
- In these actual results, using more folds did not produce a more consistent estimate: `k=10` had the greatest fold-to-fold variation.
